# Yerbanalytics — Baseline de clasificación (MobileNetV3 + Transfer Learning)

Detección de anomalías en plantines de yerba mate. Este notebook entrena un
**baseline** con **transfer learning** sobre **MobileNetV3-Large** (TensorFlow/Keras),
usando datasets donantes curados (té, café) como sustituto de yerba real.

**Clases (las que hoy tienen donante):**

| Clase | Donante |
|---|---|
| `Sano` | Tea Leaf Dataset — Healthy |
| `Clorosis` | CoLeaf — deficiencias N/Fe/Mg/Mn |
| `Dano_biotico` | Tea Leaf Dataset — 5 enfermedades |
| `Estres_solar` | Tea Leaf Diseases — Sunlight Scorching |

> **Recordá:** los donantes son sólo para *training*. La validación final se hace
> contra **yerba real de Misiones**. Hay *domain gap* (fondo controlado vs. plantín
> en vivero) — el transfer learning lo tolera, pero el test honesto es con yerba.

## 1. Montar Google Drive

Los datasets viven en tu Drive (no en el repo). Montamos para acceder a ellos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuración

Todo lo que vas a tocar está acá. Si renombraste algo en tu Drive, ajustá las rutas.

- `IMG_SIZE`: MobileNetV3 trabaja bien en 224x224.
- `SAMPLES_PER_CLASS`: submuestreamos para **equilibrar** las clases. Clorosis es el
  cuello de botella (~291 imgs), así que apuntamos a un número cercano. **Mejor pocas
  parejas que un desbalance brutal.**

In [ ]:
import os, random, shutil
from pathlib import Path
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Carpeta raiz de los datasets en tu Drive (ajustar si difiere)
DRIVE_DATASETS = Path('/content/drive/MyDrive/Personal/Yerbanalytics/datasets')

# Carpeta de trabajo (rapida, local a la VM de Colab) donde armamos el dataset unificado
UNIFIED_DIR = Path('/content/dataset_unified')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SAMPLES_PER_CLASS = 300   # tope por clase para equilibrar (Clorosis ~291)
VAL_SPLIT = 0.2
EPOCHS_HEAD = 12          # fase 1: entrenar solo la cabeza
EPOCHS_FINETUNE = 8       # fase 2: fine-tuning

CLASSES = ['Sano', 'Clorosis', 'Dano_biotico', 'Estres_solar']

# De que carpetas donantes sale cada clase (rutas relativas a DRIVE_DATASETS)
SOURCES = {
    'Sano': [
        'Tea Leaf Dataset/Healthy Leaves/Healthy_leaves',
    ],
    'Clorosis': [
        'CoLeaf/iron-Fe', 'CoLeaf/magnesium-Mg',
        'CoLeaf/manganese-Mn', 'CoLeaf/nitrogen-N',
    ],
    'Dano_biotico': [
        'Tea Leaf Dataset/Diseased Leaves/Blister_Blight',
        'Tea Leaf Dataset/Diseased Leaves/Brown_Blight',
        'Tea Leaf Dataset/Diseased Leaves/Leaf_Red_Rust',
        'Tea Leaf Dataset/Diseased Leaves/Red_Spider_Mite',
        'Tea Leaf Dataset/Diseased Leaves/Tea_Mosquito_Bug',
    ],
    'Estres_solar': [
        'Tea Leaf Diseases Dataset Towards Accurate Field Diagnosis Using Image-Based Detection/Research Dataset/Raw Dataset/Sunlight Scorching',
    ],
}

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 3. Armar el dataset unificado por clase

Los donantes vienen en estructuras distintas. Acá los **traducimos a NUESTRAS clases**:
juntamos las imagenes de cada clase, las mezclamos y tomamos hasta `SAMPLES_PER_CLASS`
para equilibrar. Quedan en carpetas `Sano/ Clorosis/ ...` — la **Forma A** (carpeta = etiqueta),
que es la que `image_dataset_from_directory` lee sola.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png'}

def list_images(folder: Path):
    if not folder.exists():
        print(f'  AVISO: no existe {folder}')
        return []
    return [p for p in folder.rglob('*') if p.suffix.lower() in IMG_EXT]

# Reset de la carpeta unificada
if UNIFIED_DIR.exists():
    shutil.rmtree(UNIFIED_DIR)

resumen = {}
for clase in CLASSES:
    candidatos = []
    for sub in SOURCES[clase]:
        candidatos += list_images(DRIVE_DATASETS / sub)
    random.shuffle(candidatos)
    elegidas = candidatos[:SAMPLES_PER_CLASS]

    destino = UNIFIED_DIR / clase
    destino.mkdir(parents=True, exist_ok=True)
    for i, src in enumerate(elegidas):
        shutil.copy(src, destino / f'{clase}_{i:04d}{src.suffix.lower()}')
    resumen[clase] = (len(candidatos), len(elegidas))

print('clase            disponibles  usadas')
for c, (disp, usa) in resumen.items():
    print(f'{c:16} {disp:>10}  {usa:>5}')

## 4. Cargar como `tf.data` (split train / validation)

`image_dataset_from_directory` infiere las etiquetas del nombre de carpeta y arma el
split. `label_mode='int'` → etiquetas enteras (usamos `SparseCategoricalCrossentropy`).
`prefetch` solapa carga y computo para que la GPU no espere.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    UNIFIED_DIR, validation_split=VAL_SPLIT, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')

val_ds = tf.keras.utils.image_dataset_from_directory(
    UNIFIED_DIR, validation_split=VAL_SPLIT, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print('Clases (en orden):', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 5. Sanity check — ver un batch

Antes de entrenar, **mira los datos con tus ojos**. Si las etiquetas no cuadran con
las imagenes, no hay modelo que valga.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 9))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i].numpy()])
        plt.axis('off')
plt.tight_layout(); plt.show()

## 6. Data augmentation

Generamos variantes (flip, rotacion, zoom) para que el modelo no memorice y generalice
mejor — clave porque tenemos **pocas imagenes por clase**. Solo se aplica en training.

> **Gotcha de MobileNetV3:** en Keras el modelo **ya incluye el preprocessing**
> (normalizacion) cuando `include_preprocessing=True` (por defecto). Por eso le pasamos
> las imagenes en rango **[0, 255]** tal cual — **no** hay que reescalar a mano.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
], name='data_augmentation')

## 7. Construir el modelo (MobileNetV3-Large)

**Transfer learning:** tomamos MobileNetV3 ya entrenado en ImageNet (sabe ver bordes,
texturas, formas), le **congelamos** el cuerpo (`trainable = False`) y le ponemos una
**cabeza nueva** para nuestras clases. Asi reusamos lo aprendido y entrenamos poquito.

In [ ]:
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=IMG_SIZE + (3,), include_top=False,
    weights='imagenet', include_preprocessing=True)
base_model.trainable = False   # fase 1: cuerpo congelado

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

## 8. Compilar y entrenar — Fase 1 (cabeza)

`EarlyStopping` corta si la validacion deja de mejorar y restaura los mejores pesos.
`ModelCheckpoint` guarda el mejor modelo en el Drive por las dudas.

> Si el desbalance siguiera molestando, aca podrias pasar `class_weight=...` a `.fit()`.
> Como ya submuestreamos para equilibrar, arrancamos sin eso.

In [ ]:
CKPT_DIR = Path('/content/drive/MyDrive/Personal/Yerbanalytics/modelos')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4,
                                     restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(str(CKPT_DIR / 'best_head.keras'),
                                       monitor='val_accuracy', save_best_only=True),
]

hist_head = model.fit(train_ds, validation_data=val_ds,
                      epochs=EPOCHS_HEAD, callbacks=callbacks)

## 9. Fine-tuning — Fase 2 (descongelar capas superiores)

Ahora descongelamos las **capas de arriba** del cuerpo (las que aprenden rasgos
especificos) y reentrenamos con un **learning rate bajito**, para ajustar sin destruir
lo aprendido en ImageNet. Las capas de abajo (rasgos genericos) las dejamos quietas.

In [ ]:
base_model.trainable = True
# Congelar todo menos las ultimas ~40 capas
FINE_TUNE_AT = len(base_model.layers) - 40
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),   # LR 100x mas chico
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

hist_ft = model.fit(train_ds, validation_data=val_ds,
                    epochs=EPOCHS_FINETUNE, callbacks=callbacks)

## 10. Curvas de entrenamiento

Si la curva de validacion se separa mucho de la de training → **overfitting**
(memorizo en vez de aprender). Con pocas imagenes es el riesgo numero uno.

In [ ]:
def juntar(h1, h2, key):
    return h1.history[key] + h2.history[key]

acc = juntar(hist_head, hist_ft, 'accuracy')
val_acc = juntar(hist_head, hist_ft, 'val_accuracy')
loss = juntar(hist_head, hist_ft, 'loss')
val_loss = juntar(hist_head, hist_ft, 'val_loss')

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='train'); plt.plot(val_acc, label='val')
plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(loss, label='train'); plt.plot(val_loss, label='val')
plt.title('Loss'); plt.legend()
plt.show()

## 11. Evaluacion — matriz de confusion y reporte

La accuracy sola engania. La **matriz de confusion** te dice *que* clases se confunden
entre si. Mira especialmente si **Clorosis** se mezcla con **Estres_solar** o con
**Dano_biotico** (amarilleos/necrosis parecidos) — ese era el riesgo que veniamos marcando.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools

y_true, y_pred = [], []
for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    y_pred += list(np.argmax(probs, axis=1))
    y_true += list(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.title('Matriz de confusion'); plt.colorbar()
ticks = range(NUM_CLASSES)
plt.xticks(ticks, class_names, rotation=45, ha='right'); plt.yticks(ticks, class_names)
for i, j in itertools.product(ticks, ticks):
    plt.text(j, i, cm[i, j], ha='center',
             color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.ylabel('Real'); plt.xlabel('Prediccion'); plt.tight_layout(); plt.show()

## 12. Guardar el modelo y exportar a TFLite (edge)

Guardamos el modelo entrenado y lo convertimos a **TFLite cuantizado** — el formato
liviano para correr en el **edge** (el gantry). La cuantizacion baja el tamano y acelera
la inferencia con una perdida minima de precision. **Esta era la ventaja de TF que charlamos.**

In [ ]:
# Modelo completo (Keras)
model.save(str(CKPT_DIR / 'yerbanalytics_baseline.keras'))

# Export a TFLite con cuantizacion (para edge)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = CKPT_DIR / 'yerbanalytics_baseline.tflite'
tflite_path.write_bytes(tflite_model)
print('Guardado:', tflite_path, f'({len(tflite_model)/1e6:.2f} MB)')

## Proximos pasos / notas

- **Curar a ojo** las clases ruidosas antes de subir `SAMPLES_PER_CLASS` (ej. descartar
  fotos de acaro/scorching donde el sintoma no se ve).
- **Sumar RoCoLe** (acaro de cafe con fondo de campo real) a `Dano_biotico`. Como RoCoLe
  etiqueta por **CSV** y no por carpetas, necesita un paso extra: leer
  `Annotations/RoCoLE-csv.csv`, filtrar `classification == red_spider_mite` y copiar esas
  imagenes a `Dano_biotico/`. Aporta robustez de fondo que al Tea Leaf le falta.
- **Validar contra yerba real** de Misiones cuando este disponible (julio).
- Probar `class_weight` o *focal loss* si reaparece el desbalance.
- Una vez solido el baseline de 4 clases, evaluar la 5ta clase del MVP (sin donante aun).